In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Modules and Model Construction

The networks we have trained so far were small enough to write down layer by
layer. The networks ahead are not: deep ResNets chain more than a hundred
convolutional layers [@He.Zhang.Ren.ea.2016], and a GPT-style language
model stacks dozens of identical Transformer blocks
[@Radford.Wu.Child.ea.2019]. Such models are assembled from repeated
*blocks*, groups of layers treated as units of design. The abstraction that
makes blocks composable is the *module*.

A module is an object with three responsibilities: it owns *parameters*, it
owns *child modules*, and it implements a *forward computation* that maps
inputs to outputs. The definition is deliberately recursive. A fully connected
layer is a module (parameters, no children). A residual block is a module (no
direct parameters, a few child layers that own parameters). A hundred-layer
network is a module whose children are blocks whose children are layers. Most
models therefore have a tree-shaped module hierarchy, as sketched in
the figure. Reusing one child at several sites turns that tree into
an object graph, a case we meet when tying parameters in
that section. Almost everything
this chapter does to a model, listing its parameters
(that section), moving it to a GPU (that section),
saving it to disk (that section), is implemented as a walk over
that tree.

![Layers compose into blocks and blocks compose into models, giving the usual tree-shaped module hierarchy.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-tree.svg)

In [ ]:
from dataclasses import dataclass
from d2l import jax as d2l
import jax
from jax import numpy as jnp
from flax import nnx

## The Module Abstraction

In JAX the module class is Flax's `nnx.Module`. We have used one of its
subclasses all along: `nnx.Sequential` builds a model from a chain of layers,
here the familiar MLP with a 256-unit ReLU hidden layer and a 10-unit output
layer. NNX modules own their initialized parameters, so construction requires
the layer widths and an RNG stream; a forward pass then uses the same direct
`net(X)` notation as the other frameworks.

In [ ]:
net = nnx.Sequential(nnx.Linear(20, 256, rngs=nnx.Rngs(0)), nnx.relu,
                     nnx.Linear(256, 10, rngs=nnx.Rngs(1)))

X = jax.random.uniform(d2l.get_key(), (2, 20))
net(X).shape

`Sequential` is not a special construct. It is itself a module whose forward
computation runs its children in order, and the children are the three modules
we passed to it. Each child owns its parameters; selecting `nnx.Param` state
gives a flat path view whose structure mirrors the module tree:

In [ ]:
[(path, value.shape)
 for path, value in nnx.state(net, nnx.Param).flat_state()]

`nnx.state(net, nnx.Param)` selects the trainable leaves of the object graph.
Optimizer updates, device movement, and serialization traverse this state.
The parameters remain owned by their layers; filters give us a state view
without maintaining a second model representation by hand.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, rngs):
        self.hidden = nnx.Linear(20, 256, rngs=rngs)
        self.out = nnx.Linear(256, 10, rngs=rngs)

    def __call__(self, X):
        return self.out(nnx.relu(self.hidden(X)))

In [ ]:
net = MLP(nnx.Rngs(0))
net(X).shape

Two details do the work here. First, assigning `nnx.Linear` children to
`self.hidden` and `self.out` makes them part of the NNX object graph, and their
attribute names become paths in the selected parameter state. Second, we never
wrote a backward method; `nnx.grad` derives gradients from whatever `__call__`
computes. Layers are constructed once in `__init__`, keeping parameter
creation separate from the forward computation.

## Sequential and Friends: Containers

To see that there is no magic left in `nnx.Sequential`, we can write it
ourselves. Two ingredients suffice: declare a field that holds the list of
children, and loop over them in `__call__`.

In [ ]:
class MySequential(nnx.Module):
    def __init__(self, *modules):
        self.modules = nnx.List(modules)

    def __call__(self, X):
        for module in self.modules:
            X = module(X)
        return X

`nnx.List` is a graph-aware container. Both `Linear` children are tracked
(`nnx.relu` is a plain function, with nothing to track), and their parameters
appear beneath the `modules` path. Our version is a drop-in replacement:

In [ ]:
net = MySequential(nnx.Linear(20, 256, rngs=nnx.Rngs(0)), nnx.relu,
                   nnx.Linear(256, 10, rngs=nnx.Rngs(1)))
net(X).shape

Graph-aware containers make registration explicit in NNX. `nnx.List` tracks
the child modules and still supports ordinary indexing and iteration:

In [ ]:
class ListMLP(nnx.Module):
    def __init__(self, rngs):
        self.layers = nnx.List([nnx.Linear(20, 256, rngs=rngs),
                                nnx.Linear(256, 10, rngs=rngs)])

    def __call__(self, X):
        return self.layers[1](nnx.relu(self.layers[0](X)))

net = ListMLP(nnx.Rngs(0))
state = nnx.state(net, nnx.Param)
net(X).shape, sum(x.size for _, x in state.flat_state())

All 7946 parameters are present. NNX deliberately rejects a plain Python list
that contains graph nodes, because treating such a container as static would
silently hide its parameters. The error points directly to `nnx.List` (or
`nnx.data`) as the repair:

In [ ]:
class PlainListSequential(nnx.Module):
    def __init__(self, *modules):
        self.modules = list(modules)  # Graph nodes cannot be static data

try:
    net = PlainListSequential(nnx.Linear(20, 10, rngs=nnx.Rngs(0)))
except ValueError as e:
    print(e)

NNX supplies `nnx.List` and `nnx.Dict` for graph nodes. A misdeclared child
fails at construction time instead of yielding a model that runs but trains
nothing. Transformer implementations conventionally keep their stack of
blocks in an `nnx.List`, with the embedding and output head as named
attributes.

## Forward Is Just Python

`__call__` is an ordinary Python method. Nothing restricts it to chaining
children: it can branch, loop, call any `jnp` function, and combine
intermediate results however it likes. The loop in `MySequential` already used
this freedom. Its most consequential one-line use is the *residual
connection*, the wiring idiom at the heart of ResNets and Transformers alike.
The residual body is constructed once and then reused on every call.

In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(self, num_hiddens, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

![The residual wiring `X + body(X)`: the input splits at a branch point into the body stack and an identity skip, and the two rejoin by addition before the block's output.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

`X + body(X)` is not a layer any framework provides. It is plain arithmetic
in the forward computation, and it changes what the block *is*: the block
computes a perturbation of the identity function rather than an arbitrary
transformation. If its body is $F$, its Jacobian is $I + J_F$, so
backpropagation receives an additive identity contribution along the skip
path. This contribution can still cancel against $J_F$; it is a direct path,
not a guarantee that every gradient is preserved
[@He.Zhang.Ren.ea.2016]. the figure diagrams
exactly this wiring, and that section develops both points when we
build ResNet; for now we only need the mechanics. One mechanical consequence
is visible already: the addition forces the input and output shapes to
agree, so a residual block has a single width that is part of its identity.

That is why `num_hiddens` is an explicit constructor argument rather than a
width inferred from a sample batch.

In [ ]:
block = ResidualBlock(24, nnx.Rngs(0))
X24 = jax.random.normal(d2l.get_key(), (2, 24))
block(X24).shape

`__call__` may also use state that is neither an input nor a parameter.
Suppose we want to damp each block's contribution by a fixed factor:

In [ ]:
class ScaledResidual(ResidualBlock):
    def __init__(self, num_hiddens, rngs, alpha=0.5):
        super().__init__(num_hiddens, rngs)
        self.alpha = alpha  # Fixed architecture data, never trained

    def __call__(self, X):
        return X + self.alpha * self.body(X)

block = ScaledResidual(24, nnx.Rngs(0))
block.alpha, [path for path, _ in nnx.state(block, nnx.Param).flat_state()]

`alpha` enters the computation, but it is ordinary architecture data rather
than an `nnx.Param`, so the optimizer never touches it. Constructing
`ScaledResidual(24, nnx.Rngs(0), alpha=0.5)` reproduces it. Mutable non-parameter state,
such as running statistics, instead uses another `nnx.Variable` subclass,
introduced in that section.

In [ ]:
class HalvingMLP(nnx.Module):
    def __init__(self, rngs):
        self.proj = nnx.Linear(20, 24, rngs=rngs)

    def __call__(self, X):
        X = self.proj(X)
        while jnp.abs(X).sum() > 1:  # Ordinary Python control flow
            X = X / 2
        return X.sum()

net = HalvingMLP(nnx.Rngs(0))
net(X)

Once a model is wrapped in `jax.jit` for speed, such data-dependent Python
control flow must be expressed with `jax.lax` primitives instead; that
constraint arrives only with the compiler, not with the module abstraction.

## Lazy Initialization: Shapes from Data

NNX makes a different choice: `nnx.Linear` takes both its input and output
widths and creates its parameters in the constructor. This keeps the model
object complete: an optimizer or checkpoint can inspect every parameter
immediately, without a dummy forward pass. The widths also make a bad
connection easier to diagnose, although an ordinary builder does not validate
adjacent layers; an incompatible pair still fails when the model is called.

There is no uninitialized form to inspect. We state the two boundary widths
and the hidden width once; the resulting state is available immediately:

In [ ]:
net = nnx.Sequential(nnx.Linear(20, 256, rngs=nnx.Rngs(0)), nnx.relu,
                     nnx.Linear(256, 10, rngs=nnx.Rngs(1)))
[(path, value.shape)
 for path, value in nnx.state(net, nnx.Param).flat_state()]

Lazy layers can remove shape arithmetic from model definitions. In a
convolutional network the flattened feature-map size depends on the input
resolution and upstream strides. We usually avoid that fragile arithmetic by
using global or adaptive pooling before the first linear layer. When a width
is part of the architecture, we name it in the model config and pass it to
adjacent layers — code that stays explicit and compact without hiding
parameter creation behind a sample batch, whether or not the framework
offers a lazy mode.

Parameters are available as soon as construction returns. Initialization
randomness comes from the `nnx.Rngs` object passed to the constructors, so a
layer's random stream is explicit (that section returns to seeding).

Each `nnx.Linear` accepts a `kernel_init` function and invokes it in the
constructor. A small demonstration:

In [ ]:
class TinyMLP(d2l.Module):
    def __init__(self, rngs):
        super().__init__()
        self.net = nnx.Sequential(
            nnx.Linear(20, 256,
                       kernel_init=nnx.initializers.xavier_uniform(),
                       rngs=rngs), nnx.relu,
            nnx.Linear(256, 10, rngs=rngs))

model = TinyMLP(nnx.Rngs(0))
[(path, value.shape)
 for path, value in nnx.state(model, nnx.Param).flat_state()]

The constructor created every parameter and drew the first kernel from the
Xavier initializer. Which
initializer to use, and why Xavier's variance rule (that section)
is a sensible default, is the subject of that section.

## Building from a Config

So far every width in this section was a literal typed into a constructor.
Real model code does not work that way. The architecture choices that vary
across experiments, such as depth, width, and output size, must be logged with
results and stored with checkpoints so that a saved model can be rebuilt. The
topology remains in code; a small configuration object records its variable
choices:

In [ ]:
@dataclass
class MLPConfig:
    d_in: int = 784
    d_hidden: int = 256
    num_blocks: int = 4
    d_out: int = 10

class ResidualMLP(nnx.Module):
    def __init__(self, cfg, rngs):
        self.cfg = cfg
        self.proj = nnx.Linear(cfg.d_in, cfg.d_hidden, rngs=rngs)
        self.blocks = nnx.List([
            ResidualBlock(cfg.d_hidden, rngs) for _ in range(cfg.num_blocks)])
        self.out = nnx.Linear(cfg.d_hidden, cfg.d_out, rngs=rngs)

    def __call__(self, X):
        X = self.proj(X)
        for block in self.blocks:
            X = block(X)
        return self.out(X)

def build(cfg: MLPConfig) -> nnx.Module:
    return ResidualMLP(cfg, nnx.Rngs(0))

One config produces one architecture. The input width is explicit because
NNX creates parameters in the constructor. `nnx.List` registers the repeated
residual blocks, while the loop in `__call__` controls their execution.
Selecting parameter state displays the resulting object graph:

In [ ]:
net = build(MLPConfig())
[(path, value.shape)
 for path, value in nnx.state(net, nnx.Param).flat_state()]

In [ ]:
net(jax.random.uniform(d2l.get_key(), (2, 784))).shape

The variable architecture choices are now data. Rescaling the model is a change to two fields, not
an edit to model code:

In [ ]:
for cfg in (MLPConfig(), MLPConfig(d_hidden=512, num_blocks=8)):
    net = build(cfg)
    n = sum(x.size for _, x in nnx.state(net, nnx.Param).flat_state())
    print(f'd_hidden={cfg.d_hidden}, num_blocks={cfg.num_blocks}: '
          f'{n:,} parameters')

Because `build` is deterministic in `cfg`, the config is all we need to
reconstruct the object graph later; that section saves it
alongside the state. Width and depth fields feeding a loop that stacks
identical residual blocks form a common construction pattern in the
Transformer models later in the book.

## Summary

A module owns parameters, child modules, and a `__call__` method. Layers,
blocks, and whole models are the same kind of object, and state selection,
optimizer updates, and serialization traverse the NNX object graph.
`nnx.List` and `nnx.Dict` register collections of children; a plain container
holding graph nodes is rejected. `__call__` is ordinary Python, so a residual
connection is one line. NNX linear layers take both widths and create their
parameters in the constructor. A dataclass config plus a deterministic build
function records those widths and repeated-block counts.

## Exercises

1. Extend `ListMLP` with an `nnx.Dict` of submodules and verify that every
   parameter appears in `nnx.state(net, nnx.Param)`. Then replace it with a
   plain dictionary holding those modules and describe how NNX reports the
   invalid graph container.
1. Implement a `ParallelBlock` that takes two child modules `net1` and `net2`
   as fields, runs both on the same input, and concatenates their outputs
   along the last dimension. What must be true of the two children's outputs
   for the concatenation to be valid?
1. Extend `MLPConfig` with an activation switch (for example,
   `act: str = 'relu'`) and make `ResidualMLP` honor it. Which decisions belong
   in the config and which belong in code? Where would you put a choice between
   `ResidualBlock` and a plain feed-forward block?
1. `ResidualBlock` requires its input and output widths to agree. Suppose you
   want a block whose output is wider than its input. Give two standard fixes
   and the cost of each. (that section uses one of them in ResNet.)